<a href="https://colab.research.google.com/github/Rixdorfer-Kindl/protein-design-pipline-colab/blob/main/rfd3_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from google.colab import drive
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
assert os.path.ismount("/content/drive"), "❌ Drive 没挂载上，先别往下跑"
print("✅ Drive 已挂载")

In [1]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
!python --version
import torch
print("torch", torch.__version__, "| cuda", torch.version.cuda, "| bf16支持:", torch.cuda.is_bf16_supported())

name, driver_version, memory.total [MiB]
Tesla T4, 580.82.07, 15360 MiB
Python 3.13.16
torch 2.11.0+cu130 | cuda 13.0 | bf16支持: True


In [4]:
#@title 安装 RFD3 并准备 6ARU 输入 { display-mode: "form" }
!pip install -q "rc-foundry[rfd3]"
!foundry install rfd3
!pip install -q biopython

import os, glob, json, numpy as np
from Bio.PDB import PDBParser, PDBIO, PDBList, Select

#@markdown ### target 与 binder 设置
dom_start = 310 #@param {type:"integer"}
dom_end = 481 #@param {type:"integer"}
binder_len_min = 60 #@param {type:"integer"}
binder_len_max = 80 #@param {type:"integer"}
# 值是该残基上希望 binder 接触的原子名，多个用逗号隔开（原子名需存在，下面会检查）
hotspots = {"A467": "CD1", "A468": "OG", "A471": "CA", "A472": "CD,OE1"}

path = PDBList().retrieve_pdb_file("6aru", pdir="/content", file_format="pdb")
s = PDBParser(QUIET=True).get_structure("s", path)

class Keep(Select):
    def accept_chain(self, c): return c.id == "A"
    def accept_residue(self, r): return r.id[0] == " " and dom_start <= r.id[1] <= dom_end
io = PDBIO(); io.set_structure(s); io.save("/content/target_A.pdb", Keep())

ch = PDBParser(QUIET=True).get_structure("t", "/content/target_A.pdb")[0]["A"]
nums = [r.id[1] for r in ch]
print("残基数:", len(nums), "| 编号范围:", nums[0], "-", nums[-1])
gaps = [(a, b) for a, b in zip(nums, nums[1:]) if b != a + 1]
print("编号不连续处（缺失残基）:", gaps or "无")

for rn, atoms in hotspots.items():
    for a in atoms.split(","):
        assert a in ch[int(rn[1:])], f"残基 {rn} 没有原子 {a}"

# 仅在无缺失时使用单段 contig；有缺失就先停下来看
assert not gaps, "target 有缺失残基，contig 需要拆段，先把上面的输出贴给我"
n_t = len(nums)
spec = {"egfr_d3": {
    "input": "/content/target_A.pdb",
    "contig": f"A{nums[0]}-{nums[-1]},/0,{binder_len_min}-{binder_len_max}",
    "length": f"{n_t+binder_len_min}-{n_t+binder_len_max}",
    "select_hotspots": hotspots,
}}
json.dump(spec, open("/content/spec.json", "w"), indent=2)
print(json.dumps(spec, indent=2))
print("权重文件:", glob.glob(os.path.expanduser("~/.foundry/checkpoints/*")))

Structure exists: '/content/pdb6aru.ent' 
残基数: 172 | 编号范围: 310 - 481
编号不连续处（缺失残基）: 无
{
  "egfr_d3": {
    "input": "/content/target_A.pdb",
    "contig": "A310-481,/0,60-80",
    "length": "232-252",
    "select_hotspots": {
      "A467": "CD1",
      "A468": "OG",
      "A471": "CA",
      "A472": "CD,OE1"
    }
  }
}
权重文件: ['/root/.foundry/checkpoints/rfd3_latest.ckpt']


In [5]:
!rfd3 design --help | head -60

                                                                                
 Usage: rfd3 [OPTIONS]                                                          
                                                                                
 Run design using hydra config overrides and input files.                       
                                                                                
╭─ Options ────────────────────────────────────────────────────────────────────╮
│ --install-completion          Install completion for the current shell.      │
│ --show-completion             Show completion for the current shell, to copy │
│                               it or customize the installation.              │
│ --help                        Show this message and exit.                    │
╰──────────────────────────────────────────────────────────────────────────────╯



In [9]:
import glob, os
ckpt = sorted(glob.glob(os.path.expanduser("~/.foundry/checkpoints/rfd3*.ckpt")))[-1]
!rfd3 design out_dir=/content/drive/MyDrive/RFD3_out/run1 inputs=/content/spec.json ckpt_path={ckpt}
# 加上你找到的批量参数，例如先只出 2 个

/usr/local/lib/python3.13/dist-packages/hydra/_internal/config_loader_impl.py:216: UserWarning: provider=hydra.searchpath in main, path=configs is not available.
  warnings.warn(
Environment variable CCD_MIRROR_PATH not set. Will not be able to use function requiring this variable. To set it you may:
  (1) add the line 'export VAR_NAME=path/to/variable' to your .bashrc or .zshrc file
  (2) set it in your current shell with 'export VAR_NAME=path/to/variable'
  (3) write it to a .env file in the root of the atomworks.io repository
Environment variable PDB_MIRROR_PATH not set. Will not be able to use function requiring this variable. To set it you may:
  (1) add the line 'export VAR_NAME=path/to/variable' to your .bashrc or .zshrc file
  (2) set it in your current shell with 'export VAR_NAME=path/to/variable'
  (3) write it to a .env file in the root of the atomworks.io repository
23:23:04 DEBUG transforms: Debug mode is on
23:23:05 INFO rfd3.engine: [rank: 0] Outputs will be written to /

In [7]:
!pip list 2>/dev/null | grep -i -E "rc-foundry|atomworks|rfd3|foundry"
!pip show rc-foundry | grep -i requires
!pip check 2>&1 | grep -i -E "atomworks|foundry|rfd3"

atomworks                             3.0.4
rc-foundry                            0.1.11
Requires: assertpy, atomworks, beartype, dm-tree, einops, einx, environs, hydra-core, ipykernel, jaxtyping, lightning, loralib, opt-einsum, pandas, rich, rootutils, toolz, torch, typer, wandb, zstandard


In [8]:
!pip install -q "atomworks[ml]<3"
!pip list 2>/dev/null | grep -i -E "atomworks|rc-foundry"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
atomworks                             2.2.1
rc-foundry                            0.1.11


In [10]:
!find /content -name "*egfr_d3*" 2>/dev/null | head -30
!ls -la /content/rfd3_test2 2>/dev/null


/content/drive/MyDrive/RFD3_out/run1/spec_egfr_d3_0_model_0.json
/content/drive/MyDrive/RFD3_out/run1/spec_egfr_d3_0_model_1.cif.gz
/content/drive/MyDrive/RFD3_out/run1/spec_egfr_d3_0_model_7.cif.gz
/content/drive/MyDrive/RFD3_out/run1/spec_egfr_d3_0_model_6.json
/content/drive/MyDrive/RFD3_out/run1/spec_egfr_d3_0_model_4.json
/content/drive/MyDrive/RFD3_out/run1/spec_egfr_d3_0_model_3.cif.gz
/content/drive/MyDrive/RFD3_out/run1/spec_egfr_d3_0_model_5.json
/content/drive/MyDrive/RFD3_out/run1/spec_egfr_d3_0_model_0.cif.gz
/content/drive/MyDrive/RFD3_out/run1/spec_egfr_d3_0_model_3.json
/content/drive/MyDrive/RFD3_out/run1/spec_egfr_d3_0_model_2.json
/content/drive/MyDrive/RFD3_out/run1/spec_egfr_d3_0_model_6.cif.gz
/content/drive/MyDrive/RFD3_out/run1/spec_egfr_d3_0_model_4.cif.gz
/content/drive/MyDrive/RFD3_out/run1/spec_egfr_d3_0_model_7.json
/content/drive/MyDrive/RFD3_out/run1/spec_egfr_d3_0_model_2.cif.gz
/content/drive/MyDrive/RFD3_out/run1/spec_egfr_d3_0_model_5.cif.gz
/content/

In [11]:
import os, shutil, glob
from google.colab import drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

src = "/content/rfd3_test2"
dst = "/content/drive/MyDrive/RFD3_out/run1"
os.makedirs(dst, exist_ok=True)

files = glob.glob(f"{src}/**/*", recursive=True)
files = [f for f in files if os.path.isfile(f)]
for f in files:
    shutil.copy2(f, os.path.join(dst, os.path.basename(f)))
print(f"已复制 {len(files)} 个文件到 {dst}")
print(sorted(os.listdir(dst)))

已复制 0 个文件到 /content/drive/MyDrive/RFD3_out/run1
['spec_egfr_d3_0_model_0.cif.gz', 'spec_egfr_d3_0_model_0.json', 'spec_egfr_d3_0_model_1.cif.gz', 'spec_egfr_d3_0_model_1.json', 'spec_egfr_d3_0_model_2.cif.gz', 'spec_egfr_d3_0_model_2.json', 'spec_egfr_d3_0_model_3.cif.gz', 'spec_egfr_d3_0_model_3.json', 'spec_egfr_d3_0_model_4.cif.gz', 'spec_egfr_d3_0_model_4.json', 'spec_egfr_d3_0_model_5.cif.gz', 'spec_egfr_d3_0_model_5.json', 'spec_egfr_d3_0_model_6.cif.gz', 'spec_egfr_d3_0_model_6.json', 'spec_egfr_d3_0_model_7.cif.gz', 'spec_egfr_d3_0_model_7.json']


In [13]:
!cd /content/drive/MyDrive/RFD3_out && zip -r /content/rfd3_run1.zip run1
from google.colab import files
files.download("/content/rfd3_run1.zip")

  adding: run1/ (stored 0%)
  adding: run1/spec_egfr_d3_0_model_0.json (deflated 71%)
  adding: run1/spec_egfr_d3_0_model_1.cif.gz (deflated 2%)
  adding: run1/spec_egfr_d3_0_model_7.cif.gz (deflated 2%)
  adding: run1/spec_egfr_d3_0_model_6.json (deflated 71%)
  adding: run1/spec_egfr_d3_0_model_4.json (deflated 71%)
  adding: run1/spec_egfr_d3_0_model_3.cif.gz (deflated 2%)
  adding: run1/spec_egfr_d3_0_model_5.json (deflated 71%)
  adding: run1/spec_egfr_d3_0_model_0.cif.gz (deflated 2%)
  adding: run1/spec_egfr_d3_0_model_3.json (deflated 71%)
  adding: run1/spec_egfr_d3_0_model_2.json (deflated 71%)
  adding: run1/spec_egfr_d3_0_model_6.cif.gz (deflated 2%)
  adding: run1/spec_egfr_d3_0_model_4.cif.gz (deflated 2%)
  adding: run1/spec_egfr_d3_0_model_7.json (deflated 71%)
  adding: run1/spec_egfr_d3_0_model_2.cif.gz (deflated 2%)
  adding: run1/spec_egfr_d3_0_model_5.cif.gz (deflated 2%)
  adding: run1/spec_egfr_d3_0_model_1.json (deflated 71%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
import os, glob, gzip, json
import numpy as np
from Bio.PDB import MMCIFParser

out_dir = "/content/drive/MyDrive/RFD3_out/run1"
parser = MMCIFParser(QUIET=True)

for f in sorted(glob.glob(f"{out_dir}/*.cif.gz")):
    name = os.path.basename(f).replace(".cif.gz", "")
    with gzip.open(f, "rt") as fh:
        model = parser.get_structure("s", fh)[0]
    info = {c.id: sum(1 for r in c if r.id[0] == " ") for c in model}
    print(f"{name}: {os.path.getsize(f)/1024:.0f} KB | 各链残基数 {info}")

# 看一个 json 里有哪些字段（可能包含 RFD3 自己算的指标）
j = sorted(glob.glob(f"{out_dir}/*_model_0.json"))[0]
d = json.load(open(j))
print("\n", os.path.basename(j), "字段:", list(d.keys()))
for k, v in d.items():
    if isinstance(v, (int, float, str)) and len(str(v)) < 80:
        print(" ", k, ":", v)


spec_egfr_d3_0_model_0: 40 KB | 各链残基数 {'A': 172, 'B': 70}
spec_egfr_d3_0_model_1: 40 KB | 各链残基数 {'A': 172, 'B': 70}
spec_egfr_d3_0_model_2: 41 KB | 各链残基数 {'A': 172, 'B': 70}
spec_egfr_d3_0_model_3: 41 KB | 各链残基数 {'A': 172, 'B': 70}
spec_egfr_d3_0_model_4: 40 KB | 各链残基数 {'A': 172, 'B': 70}
spec_egfr_d3_0_model_5: 41 KB | 各链残基数 {'A': 172, 'B': 70}
spec_egfr_d3_0_model_6: 40 KB | 各链残基数 {'A': 172, 'B': 70}
spec_egfr_d3_0_model_7: 41 KB | 各链残基数 {'A': 172, 'B': 70}

 spec_egfr_d3_0_model_0.json 字段: ['diffused_index_map', 'metrics', 'specification', 'ckpt_path', 'seed']
  ckpt_path : /root/.foundry/checkpoints/rfd3_latest.ckpt


In [15]:
import json, glob, pandas as pd
out_dir = "/content/drive/MyDrive/RFD3_out/run1"
rows = []
for j in sorted(glob.glob(f"{out_dir}/*.json")):
    d = json.load(open(j))
    m = d.get("metrics", {})
    rows.append({"file": j.split("/")[-1].replace(".json", ""), **{k: v for k, v in m.items() if isinstance(v, (int, float))}})
df_m = pd.DataFrame(rows)
pd.set_option("display.width", 250, "display.max_columns", 40)
display(df_m)
print("\n字段:", list(df_m.columns))

,file,max_ca_deviation,n_chainbreaks,n_clashing.interresidue_clashes_w_sidechain,n_clashing.interresidue_clashes_w_backbone,non_loop_fraction,loop_fraction,helix_fraction,sheet_fraction,num_ss_elements,radius_of_gyration,alanine_content,glycine_content,num_residues
0,spec_egfr_d3_0_model_0,18.129269,1,2,0,0.557143,0.442857,0.557143,0.000000,3,19.562362,0.086777,0.082645,242
1,spec_egfr_d3_0_model_1,0.106656,0,0,0,0.742857,0.257143,0.742857,0.000000,4,17.172476,0.123967,0.070248,242
2,spec_egfr_d3_0_model_2,14.697787,2,6,2,0.428571,0.571429,0.428571,0.000000,3,18.679175,0.078512,0.070248,242
3,spec_egfr_d3_0_model_3,0.106656,0,0,0,0.885714,0.114286,0.885714,0.000000,3,20.226323,0.090909,0.061983,242
4,spec_egfr_d3_0_model_4,0.205724,0,11,3,0.357143,0.642857,0.357143,0.000000,2,18.233080,0.082645,0.103306,242
5,spec_egfr_d3_0_model_5,0.114143,0,0,0,0.814286,0.185714,0.728571,0.085714,4,20.485816,0.099174,0.066116,242
6,spec_egfr_d3_0_model_6,0.114910,0,0,0,0.442857,0.557143,0.442857,0.000000,2,20.596097,0.128099,0.070248,242
7,spec_egfr_d3_0_model_7,0.106656,0,0,0,0.842857,0.157143,0.757143,0.085714,4,19.251989,0.082645,0.070248,242



字段: ['file', 'max_ca_deviation', 'n_chainbreaks', 'n_clashing.interresidue_clashes_w_sidechain', 'n_clashing.interresidue_clashes_w_backbone', 'non_loop_fraction', 'loop_fraction', 'helix_fraction', 'sheet_fraction', 'num_ss_elements', 'radius_of_gyration', 'alanine_content', 'glycine_content', 'num_residues']


In [16]:
#@title 📏 RFD3 输出的界面分析 { display-mode: "form" }
!pip install -q biopython
import os, glob, gzip, numpy as np, pandas as pd
from scipy.spatial.distance import cdist
from Bio.PDB import MMCIFParser
from Bio.Data.IUPACData import protein_letters_3to1

out_dir = "/content/drive/MyDrive/RFD3_out/run1" #@param {type:"string"}
contact_cutoff = 5.0 #@param {type:"number"}
target_len = 172 #@param {type:"integer"}
target_first_resnum = 310 #@param {type:"integer"}
# 6ARU 编号 -> 预期氨基酸，用来核对编号有没有变
hotspots = {467: "I", 468: "S", 471: "G", 472: "E"}

parser = MMCIFParser(QUIET=True)

def residues(chain):
    return [r for r in chain if r.id[0] == " "]

def heavy(res_list):
    return [a for r in res_list for a in r if a.element != "H"]

rows = []
for f in sorted(glob.glob(os.path.join(out_dir, "*.cif.gz"))):
    name = os.path.basename(f).replace(".cif.gz", "")
    try:
        with gzip.open(f, "rt") as fh:
            model = parser.get_structure("s", fh)[0]
        chains = {c.id: residues(c) for c in model}
        # 靠长度区分：长度等于 target 的是 target，另一条是 binder
        t_id = min(chains, key=lambda k: abs(len(chains[k]) - target_len))
        b_id = [k for k in chains if k != t_id and len(chains[k]) > 0]
        if not b_id:
            print(f"⚠️ {name}: 只找到链 {list(chains)}，跳过"); continue
        b_id = max(b_id, key=lambda k: len(chains[k]))
        t_res, b_res = chains[t_id], chains[b_id]

        t_atoms, b_atoms = heavy(t_res), heavy(b_res)
        D = cdist(np.array([a.coord for b in [b_atoms] for a in b]),
                  np.array([a.coord for a in t_atoms]))
        hit = D < contact_cutoff
        b_int = {b_atoms[i].get_parent().id[1] for i in np.where(hit.any(1))[0]}
        t_int = {t_atoms[j].get_parent().id[1] for j in np.where(hit.any(0))[0]}

        # 找 hotspot 残基：优先按编号，编号对不上时按位置
        row = {"file": name, "target_chain": t_id, "binder_chain": b_id,
               "binder_len": len(b_res), "target_len": len(t_res),
               "min_dist": round(float(D.min()), 2),
               "n_atom_contacts": int(hit.sum()),
               "n_binder_int_res": len(b_int), "n_target_int_res": len(t_int)}
        seq = "".join(protein_letters_3to1.get(r.get_resname().capitalize(), "X") for r in b_res)
        row["binder_seq"] = seq

        by_num = {r.id[1]: r for r in t_res}
        hs_d = []
        for h, aa in hotspots.items():
            r = by_num.get(h)
            if r is None or protein_letters_3to1.get(r.get_resname().capitalize()) != aa:
                idx = h - target_first_resnum          # 按位置找
                r = t_res[idx] if 0 <= idx < len(t_res) else None
            if r is None or protein_letters_3to1.get(r.get_resname().capitalize()) != aa:
                row[f"hs{h}"] = "编号/氨基酸对不上"; hs_d.append(np.nan); continue
            idxs = [j for j, a in enumerate(t_atoms) if a.get_parent() is r]
            d = float(D[:, idxs].min()); hs_d.append(d)
            row[f"hs{h}"] = round(d, 2)
        row["hs_min_dist"] = round(float(np.nanmin(hs_d)), 2) if not np.all(np.isnan(hs_d)) else np.nan
        row["hs_touched"] = int(np.nansum(np.array(hs_d) < contact_cutoff))
        rows.append(row)
    except Exception as e:
        print(f"❌ {name}: {e}")

df = pd.DataFrame(rows)
pd.set_option("display.width", 250, "display.max_columns", 30, "display.max_colwidth", 40)
display(df.drop(columns="binder_seq"))
df.to_csv(os.path.join(out_dir, "rfd3_interface_summary.csv"), index=False)
print("💾 已保存 rfd3_interface_summary.csv")

,file,target_chain,binder_chain,binder_len,target_len,min_dist,n_atom_contacts,n_binder_int_res,n_target_int_res,hs467,hs468,hs471,hs472,hs_min_dist,hs_touched
0,spec_egfr_d3_0_model_0,A,B,70,172,1.27,915,40,49,4.56,5.19,11.49,9.37,4.56,1
1,spec_egfr_d3_0_model_1,A,B,70,172,1.71,782,39,49,4.22,3.40,9.50,7.25,3.40,2
2,spec_egfr_d3_0_model_2,A,B,70,172,0.64,1304,49,72,4.35,2.86,10.24,8.77,2.86,2
3,spec_egfr_d3_0_model_3,A,B,70,172,1.75,776,46,46,9.84,8.15,10.34,6.95,6.95,0
4,spec_egfr_d3_0_model_4,A,B,70,172,0.62,2248,49,68,9.33,7.22,10.16,8.62,7.22,0
5,spec_egfr_d3_0_model_5,A,B,70,172,1.57,843,41,57,10.37,10.14,7.17,2.76,2.76,1
6,spec_egfr_d3_0_model_6,A,B,70,172,1.66,880,38,56,9.74,8.48,9.91,8.37,8.37,0
7,spec_egfr_d3_0_model_7,A,B,70,172,1.60,757,42,63,2.70,5.11,11.98,10.05,2.70,1


💾 已保存 rfd3_interface_summary.csv


In [17]:
import os, glob, gzip, numpy as np
from scipy.spatial.distance import cdist
from Bio.PDB import MMCIFParser
out_dir = "/content/drive/MyDrive/RFD3_out/run1"
p = MMCIFParser(QUIET=True)
for f in sorted(glob.glob(f"{out_dir}/*.cif.gz")):
    with gzip.open(f, "rt") as fh:
        m = p.get_structure("s", fh)[0]
    A = [a for r in m["A"] if r.id[0]==" " for a in r]
    B = [a for r in m["B"] if r.id[0]==" " for a in r]
    D = cdist(np.array([a.coord for a in B]), np.array([a.coord for a in A]))
    i, j = np.unravel_index(D.argmin(), D.shape)
    print(os.path.basename(f)[-12:-7], f"<2.5Å的原子对: {(D<2.5).sum():4d}",
          f"最近: B:{B[i].get_parent().get_resname()}{B[i].get_parent().id[1]}-{B[i].get_id()} ~ "
          f"A:{A[j].get_parent().get_resname()}{A[j].get_parent().id[1]}-{A[j].get_id()} {D[i,j]:.2f}Å",
          "| binder残基类型:", sorted({r.get_resname() for r in m["B"]})[:6])

del_0 <2.5Å的原子对:   21 最近: B:HIS53-CE1 ~ A:VAL172-O 1.27Å | binder残基类型: ['ALA', 'ASN', 'ASP', 'GLN', 'GLU', 'GLY']
del_1 <2.5Å的原子对:    8 最近: B:ASP57-OD1 ~ A:ARG44-NH2 1.71Å | binder残基类型: ['ALA', 'ASN', 'ASP', 'GLU', 'GLY', 'HIS']
del_2 <2.5Å的原子对:   50 最近: B:SER59-O ~ A:PHE103-C 0.64Å | binder残基类型: ['ALA', 'ARG', 'ASN', 'ASP', 'CYS', 'GLN']
del_3 <2.5Å的原子对:    9 最近: B:GLU50-OE2 ~ A:HIS100-CE1 1.75Å | binder残基类型: ['ALA', 'ARG', 'ASN', 'ASP', 'CYS', 'GLN']
del_4 <2.5Å的原子对:  118 最近: B:GLY54-C ~ A:TRP77-N 0.62Å | binder残基类型: ['ALA', 'ASN', 'ASP', 'CYS', 'GLU', 'GLY']
del_5 <2.5Å的原子对:    5 最近: B:PHE5-CZ ~ A:TYR138-CB 1.57Å | binder残基类型: ['ALA', 'ASN', 'ASP', 'CYS', 'GLN', 'GLU']
del_6 <2.5Å的原子对:    7 最近: B:ASN7-O ~ A:LYS121-NZ 1.66Å | binder残基类型: ['ALA', 'ASN', 'ASP', 'GLN', 'GLU', 'GLY']
del_7 <2.5Å的原子对:    6 最近: B:LEU30-CD2 ~ A:GLN75-NE2 1.60Å | binder残基类型: ['ALA', 'ASN', 'ASP', 'GLN', 'GLU', 'GLY']


In [18]:
import os, glob, gzip, pandas as pd
from Bio.PDB import MMCIFParser, PDBParser, PDBIO, Select
from Bio.Data.IUPACData import protein_letters_3to1

out_dir = "/content/drive/MyDrive/RFD3_out/run1"
keep = [1, 5, 7]          # 只选这三个，其余先不验证
p = MMCIFParser(QUIET=True)

rows = []
for i in keep:
    f = f"{out_dir}/spec_egfr_d3_0_model_{i}.cif.gz"
    with gzip.open(f, "rt") as fh:
        m = p.get_structure("s", fh)[0]
    seq = "".join(protein_letters_3to1.get(r.get_resname().capitalize(), "X")
                  for r in m["B"] if r.id[0] == " ")
    rows.append({"backbone": "rfd3", "design": i, "seq": seq})
    print(i, len(seq), seq)

pd.DataFrame(rows).to_csv(f"{out_dir}/rfd3_candidates.csv", index=False)

1 70 SSISLDSWAAAVDTHLATGTVAAAISAAEAALNLKILKALLPAAFAELVTAALNKDDAAIAALIGKANIA
5 70 PGANFFNQDEVTALVANASAKAKTADDQVALAKFNAAADVVAVIHQAITFAALASDPDNQSLIILCSKVK
7 70 AQIAALVKNIETSSHPIGPAISNDEITEALSQLLSANLSIASLEVLSYLAALHANLAGLSVVVDTLQAAE


In [19]:
   import shutil; shutil.copy("/content/target_A.pdb", "/content/drive/MyDrive/RFD3_out/target_A.pdb")

'/content/drive/MyDrive/RFD3_out/target_A.pdb'